# Model evaluation

## 1 load data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss,
    brier_score_loss,
    confusion_matrix
)

In [2]:
df = pd.read_csv("../data/final/tournament_matchups.csv")
df.columns

Index(['YEAR', 'DayNum', 'TEAM_A', 'TEAM_B', 'TEAM_A_WIN', 'SEED_DIFF',
       '2P_O_DIFF', '2P_D_DIFF', '3P_O_DIFF', '3P_D_DIFF', 'ADJ_T_DIFF',
       'WAB_DIFF', 'KADJ_O_DIFF', 'KADJ_D_DIFF', 'BARTHAG_DIFF', 'EFG%_DIFF',
       'EFG%D_DIFF', 'TOV%_DIFF', 'TOV%D_DIFF', 'OREB%_DIFF', 'DREB%_DIFF',
       'FTR_DIFF', 'FTRD_DIFF', 'POWER_RATING_DIFF'],
      dtype='object')

## 2 select features

In [3]:
#Y target since that shows who won which game
y = df['TEAM_A_WIN']


In [4]:
X = feature_columns = [
    col for col in df.columns
    if col.endswith("_DIFF")
]

# Features
X = df[feature_columns]

# Target
y = df["TEAM_A_WIN"]

In [5]:
X.columns

Index(['SEED_DIFF', '2P_O_DIFF', '2P_D_DIFF', '3P_O_DIFF', '3P_D_DIFF',
       'ADJ_T_DIFF', 'WAB_DIFF', 'KADJ_O_DIFF', 'KADJ_D_DIFF', 'BARTHAG_DIFF',
       'EFG%_DIFF', 'EFG%D_DIFF', 'TOV%_DIFF', 'TOV%D_DIFF', 'OREB%_DIFF',
       'DREB%_DIFF', 'FTR_DIFF', 'FTRD_DIFF', 'POWER_RATING_DIFF'],
      dtype='object')

In [6]:
print(y)

0      0
1      1
2      1
3      0
4      0
      ..
530    1
531    0
532    0
533    0
534    1
Name: TEAM_A_WIN, Length: 535, dtype: int64


## 3 Create Chronological Train/Test Split

In [7]:
#training on all matchups before 2024 and testing on 2024
X_train = X[df["YEAR"] < 2024]
X_test = X[df["YEAR"] == 2024]
y_train = y[df["YEAR"] < 2024]
y_test = y[df["YEAR"] == 2024]

In [8]:
#testing
print("Training years:", sorted(df.loc[df["YEAR"] < 2024, "YEAR"].unique()))
print("Testing years:", sorted(df.loc[df["YEAR"] == 2024, "YEAR"].unique()))

Training years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023)]
Testing years: [np.int64(2024)]


## 4 seed baseline

### 4.1 create seed predictions

In [9]:
# since higer seeds are negative doing seed baseline that higher seeds win games so if seed_dif < 0 then they were a better team and should win 
seed_predictions = (X_test["SEED_DIFF"] < 0).astype(int)

### 4.2 test seed predictions

In [10]:
#testing against acual data
seed_accuracy = accuracy_score(y_test, seed_predictions)

print(f"Seed Baseline Accuracy: {seed_accuracy:.2%}") #65.67% 

Seed Baseline Accuracy: 65.67%


## 5  Logistic Regression

### 5.1 Scale the Features

##### standadizing features scales

In [11]:
from sklearn.preprocessing import StandardScaler

In [12]:
scaler = StandardScaler()

In [13]:
X_train_scaled = scaler.fit_transform(X_train)

In [14]:
X_test_scaled = scaler.transform(X_test)

In [15]:
#check
print("Original training shape:", X_train.shape)
print("Scaled training shape:", X_train_scaled.shape)

print("Original testing shape:", X_test.shape)
print("Scaled testing shape:", X_test_scaled.shape)

Original training shape: (468, 19)
Scaled training shape: (468, 19)
Original testing shape: (67, 19)
Scaled testing shape: (67, 19)


### 5.2 Create and train Logistic Regression Model

In [16]:
logisticRegr = LogisticRegression()

In [17]:
logisticRegr.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


### 5.3 predict 2024 winners

In [18]:
predictions = logisticRegr.predict(X_test_scaled)

In [19]:
print(predictions)

[1 0 1 0 0 0 1 0 1 0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 1 1 1 0 1 0 1 0 0 0 0 0 0
 0 0 0 0 0 0 1 1 0 1 0 1 0 0 1 0 1 0 0 1 1 1 0 1 1 0 0 0 0 1]


## 5.4 Predict 2024 Win Probabilities

In [20]:
win_probabilities = logisticRegr.predict_proba(X_test_scaled)[:, 1]

In [21]:
print(win_probabilities[:10])

[0.61023902 0.49948892 0.58319521 0.41193223 0.07993625 0.15659693
 0.62138166 0.27228746 0.65349062 0.21472089]


### 5.5 Calculate Logistic Regression Accuracy

In [22]:
logistic_accuracy = accuracy_score(y_test, predictions)

print(f"Logistic Regression Accuracy: {logistic_accuracy:.2%}")
print(f"Seed Baseline Accuracy: {seed_accuracy:.2%}")

Logistic Regression Accuracy: 68.66%
Seed Baseline Accuracy: 65.67%


### 5.6 Evaluate Probability Quality

In [23]:
logistic_roc_auc = roc_auc_score(y_test, win_probabilities)
logistic_logloss = log_loss(y_test, win_probabilities)
logistic_brier = brier_score_loss(y_test, win_probabilities)

print(f"ROC-AUC: {logistic_roc_auc:.3f}")
print(f"Log Loss: {logistic_logloss:.3f}")
print(f"Brier Score: {logistic_brier:.3f}")

ROC-AUC: 0.780
Log Loss: 0.570
Brier Score: 0.198


### 5.7 examine model coefficents

In [24]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logisticRegr.coef_[0]
})

In [25]:
coefficients["Absolute_Coefficient"] = coefficients["Coefficient"].abs()

In [26]:
coefficients = coefficients.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

coefficients

,Feature,Coefficient,Absolute_Coefficient
11,EFG%D_DIFF,1.221009,1.221009
18,POWER_RATING_DIFF,1.127971,1.127971
2,2P_D_DIFF,-0.907105,0.907105
4,3P_D_DIFF,-0.712475,0.712475
7,KADJ_O_DIFF,0.404937,0.404937
0,SEED_DIFF,0.378230,0.378230
6,WAB_DIFF,0.310025,0.310025
13,TOV%D_DIFF,0.207295,0.207295
16,FTR_DIFF,-0.203221,0.203221
15,DREB%_DIFF,-0.162407,0.162407


### 5.8 Logistic Regression Interpretation

##### The logistic regression model achieved an accuracy of 68.66% on the 2024 tournament test set, compared with 65.67% for the seed baseline. This was an improvement of 2.99 percentage points.
##### The model had a ROC-AUC of 0.780, a log loss of 0.570, and a Brier score of 0.198.
##### The largest absolute coefficients were EFG%D_DIFF, POWER_RATING_DIFF, 2P_D_DIFF, 3P_D_DIFF, and KADJ_O_DIFF. These were the features with yhe largest assosiation with the models predictions, seed diff was also an important feature but it didnt have a very high assosiation, EFG%D_DIFF is the most important factor in winning
##### Overall the Logistic regression model outperformed just picking the higher seed team to win everytime, the 2024 seasson did only contain 67 same so aditional models and testing across more years are needed to evaluate how well the model generalizes.

## 6 Random forest

### 6.1 create model

In [27]:
random_forest = RandomForestClassifier(n_estimators=100,random_state=42,oob_score=True)



In [28]:
random_forest.fit(X_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


### 6.2 predict 2024 winners

In [29]:
random_forest_predictions = random_forest.predict(X_test)
print(random_forest_predictions)

[1 1 1 0 0 0 1 0 1 0 1 0 0 0 0 0 1 0 1 1 1 1 0 0 1 1 1 0 1 0 1 0 0 0 0 0 0
 0 1 0 0 0 0 1 1 0 1 0 1 0 0 1 0 1 0 0 1 1 1 0 1 1 0 0 0 0 1]


## 6.3 predict 2024 winners probabilitys 

In [30]:
win_probabilities_forest = random_forest.predict_proba(X_test)[:, 1]

In [31]:
print(win_probabilities_forest[:10])

[0.61 0.56 0.8  0.41 0.03 0.24 0.65 0.34 0.57 0.18]


### 6.4Calculate forest tree  Accuracy

In [32]:
forest_accuracy = accuracy_score(y_test,random_forest_predictions )

print(f"Random Forest Accuracy: {forest_accuracy:.2%}")
print(f"Seed Baseline Accuracy: {seed_accuracy:.2%}")

Random Forest Accuracy: 68.66%
Seed Baseline Accuracy: 65.67%


In [33]:
print("Same predictions:",
      (predictions == random_forest_predictions).sum(),
      "out of",
      len(y_test))

Same predictions: 61 out of 67


### 6.5 Evaluate Probability Quality

In [34]:
forest_roc_auc = roc_auc_score(y_test, win_probabilities_forest)
forest_logloss = log_loss(y_test, win_probabilities_forest)
forest_brier = brier_score_loss(y_test, win_probabilities_forest)

print(f"Random Forest ROC-AUC: {forest_roc_auc:.3f}")
print(f"Random Forest Log Loss: {forest_logloss:.3f}")
print(f"Random Forest Brier Score: {forest_brier:.3f}")

Random Forest ROC-AUC: 0.805
Random Forest Log Loss: 0.548
Random Forest Brier Score: 0.184


### 5.7 examine model coefficents

In [35]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logisticRegr.coef_[0]
})

coefficients["Absolute_Coefficient"] = coefficients["Coefficient"].abs()



In [36]:
coefficients["Absolute_Coefficient"] = coefficients["Coefficient"].abs()


In [37]:
coefficients = coefficients.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

coefficients

,Feature,Coefficient,Absolute_Coefficient
11,EFG%D_DIFF,1.221009,1.221009
18,POWER_RATING_DIFF,1.127971,1.127971
2,2P_D_DIFF,-0.907105,0.907105
4,3P_D_DIFF,-0.712475,0.712475
7,KADJ_O_DIFF,0.404937,0.404937
0,SEED_DIFF,0.378230,0.378230
6,WAB_DIFF,0.310025,0.310025
13,TOV%D_DIFF,0.207295,0.207295
16,FTR_DIFF,-0.203221,0.203221
15,DREB%_DIFF,-0.162407,0.162407


### 6.8 Random Forest Interpretation

##### The Random Forest model achieved an accuracy of 68.66% on the 2024 tournament test set, which is the same as the logistic Regresstion model and migher thatn the  65.67% from the seed baseline. 
##### The model had a ROC-AUC of 0.780, a log loss of 0.570, and a Brier score of 0.198.
##### The Random Forest achieved a ROC-AUC of 0.805, a log loss of 0.548, and a Brier score of 0.184. Although Random Forest and Logistic Regression had the same accuracy, Random Forest performed better on all three probability-based evaluation metrics.
##### Overall the Random forest model outperformed had the same accuracy as thhe Logistic Regression model but outperformed it in the more advanced metrics, the results are only on 67 games so futher testing on more years should be evaluated in future work

## 7. Final Model Comparison


### 7.1 Create Results Table


In [38]:
model_comparison = pd.DataFrame({
    "Model": [
        "Seed Baseline",
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy": [
        seed_accuracy,
        logistic_accuracy,
        forest_accuracy
    ],

    "ROC_AUC": [
        None,
        logistic_roc_auc,
        forest_roc_auc
    ],

    "Log_Loss": [
        None,
        logistic_logloss,
        forest_logloss
    ],

    "Brier_Score": [
        None,
        logistic_brier,
        forest_brier
    ]
})

model_comparison

,Model,Accuracy,ROC_AUC,Log_Loss,Brier_Score
0,Seed Baseline,0.656716,NaN,NaN,NaN
1,Logistic Regression,0.686567,0.779857,0.570488,0.197525
2,Random Forest,0.686567,0.805258,0.548103,0.184231


### 7.2 Compare Model Performance


#### Both Logistic Regression and Random Forest achieved an accuracy of 68.66% on the 2024 tournament, outperforming the seed baseline accuracy of 65.67%.
#### Although the two machine learning models had the same accuracy, Random Forest performed better on the probability based metrics. Random Forest achieved a ROC-AUC of 0.805 compared with 0.780 for Logistic Regression. It also had a lower log loss and Brier score.
#### This suggests that while both models correctly predicted the same number of games, Random Forest produced better probability estimates on the 2024 test set.

### 7.3 Final Modeling Conclusion

#### The goal of this modeling was to predict NCAA tournament game winners using team statistics from the tournament matchups dataset 
#### both logistical regression and random forest models had an acruaccy of 68.66% accuracy on the 2024 tournament test set, compared with 65.67% for the seed-based baseline. Random Forest also produced better advanced metrics, with a ROC-AUC of 0.805, log loss of 0.548, and Brier score of 0.184. Based on the 2024 test results, Random Forest showed the strongest probability performance of the models tested. However, the evaluation contains only 67 games from one tournament, so these results are not enough to determine how consistently the models will perform across future tournaments.
#### Future improvements could include testing across multiple tournament years, tuning model hyperparameters, reconstructing win percentage using only pre-tournament games, and evaluating additional classification models.